<a href="https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

Paste your Hugging Face READ token (hf_...): ··········


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item on one report_date, within fact_content_daily_performance, restricted to the month=2026-03 partition — a mid-panel month, per the rule that _sample is the sealed final month and must never be used for label logic.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context: content_hash_id, client_hash_id, report_date

Features: gsc_avg_position, gsc_impressions, gsc_clicks, content_type, content age at report date.

Label/proxy: CTR trend within the month, split at the mid-month point.

Excluded: any product-decision flags.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Grain
con.sql(f"SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c FROM read_parquet('{MONTH_PATH}') GROUP BY report_date, client_hash_id, content_hash_id HAVING COUNT(*) > 1 LIMIT 5").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


In [ ]:
# Counts + Window
con.sql(f"SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM read_parquet('{MONTH_PATH}')").df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
# Availability, with IS TRUE
con.sql(f"SELECT COUNT(*) AS total_rows, SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows FROM read_parquet('{MONTH_PATH}')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966.0


In [ ]:
import numpy as np

con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 5").df()

features = con.sql(f"""
    WITH bounds AS (
        SELECT MIN(report_date) AS start_d FROM read_parquet('{MONTH_PATH}')
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date <= b.start_d + INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
               SUM(CASE WHEN f.report_date >  b.start_d + INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_second_half,
               SUM(CASE WHEN f.report_date <= b.start_d + INTERVAL 15 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_first_half,
               SUM(CASE WHEN f.report_date >  b.start_d + INTERVAL 15 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_second_half,
               SUM(CASE WHEN f.report_date <= b.start_d + INTERVAL 15 DAY THEN f.gsc_sum_position ELSE 0 END) AS sum_pos_first_half
        FROM read_parquet('{MONTH_PATH}') f, bounds b
        GROUP BY 1, 2
        HAVING imp_first_half >= 100
    )
    SELECT * FROM windowed
""").df()

features['pos_first_half'] = features['sum_pos_first_half'] / features['imp_first_half']
features['ctr_first_half']  = features['clk_first_half'] / features['imp_first_half']
features['ctr_second_half'] = features['clk_second_half'] / features['imp_second_half'].replace(0, np.nan)

content_meta = con.sql(f"SELECT content_hash_id, content_type, content_created_date FROM {TABLES['dim_content']}").df()
features = features.merge(content_meta, on='content_hash_id', how='left')
features['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(features['content_created_date'])).dt.days

print(f"{len(features):,} content items with enough first-half volume")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

79,572 content items with enough first-half volume


,client_hash_id,content_hash_id,imp_first_half,imp_second_half,clk_first_half,clk_second_half,sum_pos_first_half,pos_first_half,ctr_first_half,ctr_second_half,content_type,content_created_date,content_age_days
0,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,212.0,390.0,2.0,2.0,850.0,4.009434,0.009434,0.005128,keyword article,2026-02-12,47
1,client_62f4a7e64f5e0096,content_275b6f7f733016d4,501.0,309.0,1.0,0.0,2220.0,4.431138,0.001996,0.000000,keyword article,2026-02-12,47
2,client_62f4a7e64f5e0096,content_755d951187fcd70a,828.0,1030.0,2.0,4.0,1574.0,1.900966,0.002415,0.003883,keyword article,2026-02-12,47
3,client_62f4a7e64f5e0096,content_92c381fbd361212e,255.0,281.0,0.0,1.0,1256.0,4.925490,0.000000,0.003559,keyword article,2026-02-12,47
4,client_62f4a7e64f5e0096,content_97188a7032a705cf,242.0,254.0,2.0,1.0,835.0,3.450413,0.008264,0.003937,keyword article,2026-02-12,47


Five features, each knowable at the decision moment:

1. imp_first_half: only sums days before the mid-month cutoff.
2. pos_first_half: same reasoning, first-half days only.
3. content_type: static metadata, known at content creation.
4. content_age_days: pure date arithmetic, no future information.
5. ctr_first_half: trailing-window CTR from first-half impressions only.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

features['is_ctr_declining'] = (features['ctr_second_half'] < 0.8 * features['ctr_first_half']).astype(int)
model_data = features.dropna(subset=['imp_first_half','pos_first_half','content_age_days','ctr_first_half','is_ctr_declining'])

honest_cols = ['imp_first_half', 'pos_first_half', 'content_age_days', 'ctr_first_half']
X, y = model_data[honest_cols], model_data['is_ctr_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print("HONEST:")
print(classification_report(y_te, honest_model.predict(X_te), digits=3))

# THE TRAP
model_data = model_data.copy()
model_data['leaky_ctr_second_half'] = model_data['ctr_second_half']
leaky_cols = honest_cols + ['leaky_ctr_second_half']
X2, y2 = model_data[leaky_cols], model_data['is_ctr_declining']
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y2, test_size=0.25, random_state=42, stratify=y2)
leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
print("\nLEAKY (watch this jump toward perfect):")
print(classification_report(y_te2, leaky_model.predict(X_te2), digits=3))


HONEST:
              precision    recall  f1-score   support

           0      0.839     0.775     0.806     12607
           1      0.656     0.743     0.697      7286

    accuracy                          0.763     19893
   macro avg      0.748     0.759     0.751     19893
weighted avg      0.772     0.763     0.766     19893


LEAKY (watch this jump toward perfect):
              precision    recall  f1-score   support

           0      0.996     0.996     0.996     12607
           1      0.994     0.993     0.993      7286

    accuracy                          0.995     19893
   macro avg      0.995     0.995     0.995     19893
weighted avg      0.995     0.995     0.995     19893



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. A 15-day half-month split is short for a CTR comparison, so the label is noisier than a full 30-day window would give.
2. Client history is unbalanced, only 9 of 70 clients have 12+ months of data, so this March slice likely doesn't represent all clients equally, and clients who onboarded late in the panel may be thin or absent here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.